# 04 CSEE -> DeepSeek 生成 LLM 类样本

目的：基于 `CESS/CSEE_data.csv` 的人类作文，调用 DeepSeek API 生成同题 `LLM` 类文本（标签=1），用于缓解答辩中的“中文母语者英语写作习惯迁移”问题。

输出文件：
- `CESS/CSEE_llm_generated.csv`（仅 LLM 类）
- `CESS/CSEE_human_llm_balanced.csv`（Human + LLM 拼接，可直接用于训练）

使用前准备：
1. 设置环境变量：`export DEEPSEEK_API_KEY=你的key`
2. 确保可访问 DeepSeek API
3. 按顺序执行所有单元格


In [8]:
# 如需安装依赖，取消注释执行
# !pip install -q pandas requests tqdm

In [43]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


In [44]:
# ===== 配置区 =====
# 自动定位项目根目录：兼容从项目根目录或 notebooks 目录启动
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "CESS" / "CSEE_data.csv").exists():
        PROJECT_ROOT = cand
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"未找到 CESS/CSEE_data.csv，当前工作目录: {Path.cwd()}"
    )

INPUT_CSV = PROJECT_ROOT / "CESS" / "CSEE_data.csv"
OUT_LLM_CSV = PROJECT_ROOT / "CESS" / "CSEE_llm_generated.csv"
OUT_MIXED_CSV = PROJECT_ROOT / "CESS" / "CSEE_human_llm_balanced.csv"
CHECKPOINT_JSONL = PROJECT_ROOT / "CESS" / "CSEE_llm_checkpoint.jsonl"

TEXT_COL = "essay"
PROMPT_COL = "prompt"
ID_COL = "essay_id"

# AI Studio（飞桨星河社区）OpenAI兼容接口配置
# accessToken 获取: https://aistudio.baidu.com/account/accessToken
API_KEYS = [
    "9515f2dcda79e303f59b03a433db0647a71aa14a",
    "48c4caa81ed0fba9ffb50488c79e86d93e23b23b",
]
# 允许你留空某个 token
API_KEYS = [k.strip() for k in API_KEYS if k and k.strip()]

# 如果上面留空，则读取环境变量（逗号分隔多个）
if not API_KEYS:
    env_keys = os.getenv("AI_STUDIO_API_KEYS", "").strip()
    if env_keys:
        API_KEYS = [k.strip() for k in env_keys.split(",") if k.strip()]

# 向后兼容单 key变量
API_KEY = API_KEYS[0] if API_KEYS else ""

API_BASE_URL = os.getenv("API_BASE_URL", "https://aistudio.baidu.com/llm/lmapi/v3")
MODEL_NAME = os.getenv("MODEL_NAME", "ernie-3.5-8k")

# 并发控制：建议不超过 token 数量
REQUEST_WORKERS = min(2, max(1, len(API_KEYS)))

# 生成控制
MAX_SAMPLES = None          # None 表示全量；可改成整数先小规模试跑
TEMPERATURE = 0.9
MAX_NEW_TOKENS = 700
SAVE_EVERY = 50
SLEEP_BETWEEN_REQ = 0.1
MAX_RETRIES = 5
TIMEOUT = 90

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_CSV:", INPUT_CSV)
print("API_BASE_URL:", API_BASE_URL)
print("MODEL_NAME:", MODEL_NAME)
print("TOKENS:", len(API_KEYS), "WORKERS:", REQUEST_WORKERS)

assert API_KEYS, "请先填写至少一个 AI Studio token（API_KEYS）"



PROJECT_ROOT: /Users/songling/Desktop/Artificial-Intelligence-Competition---Large-Language-Model-LLM--main/LLM-Detect AI Generated Text
INPUT_CSV: /Users/songling/Desktop/Artificial-Intelligence-Competition---Large-Language-Model-LLM--main/LLM-Detect AI Generated Text/CESS/CSEE_data.csv
API_BASE_URL: https://aistudio.baidu.com/llm/lmapi/v3
MODEL_NAME: ernie-3.5-8k
TOKENS: 2 WORKERS: 2


In [45]:
df = pd.read_csv(INPUT_CSV)

# 只保留必要字段，并做最小清洗
keep_cols = [c for c in [ID_COL, PROMPT_COL, TEXT_COL, "prompt_id", "overall_score", "content_score", "language_score", "structure_score"] if c in df.columns]
df = df[keep_cols].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()
df[PROMPT_COL] = df[PROMPT_COL].fillna("").astype(str).str.strip()

df = df[(df[TEXT_COL].str.len() > 30) & (df[PROMPT_COL].str.len() > 10)].copy()
if MAX_SAMPLES:
    df = df.head(MAX_SAMPLES).copy()

df["label"] = 0
print("输入样本量:", len(df))
df.head(2)

输入样本量: 13266


,essay_id,prompt,essay,prompt_id,overall_score,content_score,language_score,structure_score,label
0,11039714,"Suppose you are Li Hua, a senior student at Ho...","Dear Jim,\nI'm glad to write to you. Our schoo...",1,5.5,2.0,2.0,1.5,0
1,11035775,"Suppose you are Li Hua, a senior student at Ho...","Dear Jim,\nI hope you are doing well in Britai...",1,12.5,5.0,4.0,3.5,0


In [46]:
def build_messages(prompt_text: str, human_essay: str):
    system_prompt = (
        "You are an assistant that writes English essays for data augmentation. "
        "Return only the final essay text with no explanation."
    )

    target_words = max(120, min(420, len(human_essay.split()) + 20))
    user_prompt = f"""
You will generate ONE English student essay for binary AI-text detection research.

Task constraints:
1) Follow this writing prompt strictly:
---
{prompt_text}
---
2) The essay should look like realistic Chinese EFL (English as a Foreign Language) student writing:
   - Natural but not perfect native style
   - May contain minor grammar/collocation issues
   - Keep coherent logic and complete structure
3) Do NOT copy the reference essay. Keep topic aligned but wording and organization new.
4) Length target: around {target_words} words.
5) No markdown, no title unless the prompt requires, no meta-commentary.

Reference human essay (for style level only, do not copy):
---
{human_essay}
---
""".strip()

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [47]:
def call_llm_api(messages, api_key=None):
    use_key = api_key or API_KEY
    url = f"{API_BASE_URL.rstrip('/')}/chat/completions"
    headers = {
        "Authorization": f"Bearer {use_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_NEW_TOKENS,
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=TIMEOUT)
            if resp.status_code == 200:
                data = resp.json()
                text = data["choices"][0]["message"]["content"].strip()
                return text, None

            if resp.status_code in (429, 500, 502, 503, 504):
                last_err = f"HTTP {resp.status_code}: {resp.text[:300]}"
                time.sleep(min(2 ** attempt, 20))
                continue

            return "", f"HTTP {resp.status_code}: {resp.text[:500]}"
        except Exception as e:
            last_err = str(e)
            time.sleep(min(2 ** attempt, 20))

    return "", last_err



In [48]:
# 断点续跑：读取已完成 checkpoint
completed = {}
if CHECKPOINT_JSONL.exists():
    with CHECKPOINT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed[item[ID_COL]] = item

print("已完成样本数:", len(completed))

todo = df[~df[ID_COL].isin(completed.keys())].copy()
print("待生成样本数:", len(todo))

已完成样本数: 40
待生成样本数: 13188


In [49]:
generated_rows = list(completed.values())

def generate_one(row, api_key):
    essay_id = getattr(row, ID_COL)
    prompt_text = getattr(row, PROMPT_COL)
    human_essay = getattr(row, TEXT_COL)

    messages = build_messages(prompt_text, human_essay)
    llm_text, err = call_llm_api(messages, api_key=api_key)

    out = {
        ID_COL: essay_id,
        "prompt_id": getattr(row, "prompt_id", None),
        PROMPT_COL: prompt_text,
        "human_essay": human_essay,
        "essay": llm_text,
        "label": 1,
        "generator": MODEL_NAME,
        "error": err,
    }
    return out

rows = list(todo.itertuples(index=False))
with CHECKPOINT_JSONL.open("a", encoding="utf-8") as fout:
    with ThreadPoolExecutor(max_workers=REQUEST_WORKERS) as ex:
        futures = []
        for i, row in enumerate(rows):
            key = API_KEYS[i % len(API_KEYS)]
            futures.append(ex.submit(generate_one, row, key))

        for i, fut in enumerate(tqdm(as_completed(futures), total=len(futures))):
            out = fut.result()
            generated_rows.append(out)
            fout.write(json.dumps(out, ensure_ascii=False) + "\n")

            if (i + 1) % SAVE_EVERY == 0:
                pd.DataFrame(generated_rows).to_csv(OUT_LLM_CSV, index=False)

            time.sleep(SLEEP_BETWEEN_REQ)

llm_df = pd.DataFrame(generated_rows)
llm_df.to_csv(OUT_LLM_CSV, index=False)
print("LLM 文件已保存:", OUT_LLM_CSV)
print("总行数:", len(llm_df))
print("失败数:", int(llm_df["error"].notna().sum()))



100%|██████████| 13188/13188 [20:08:40<00:00,  5.50s/it]  


LLM 文件已保存: /Users/songling/Desktop/Artificial-Intelligence-Competition---Large-Language-Model-LLM--main/LLM-Detect AI Generated Text/CESS/CSEE_llm_generated.csv
总行数: 13228
失败数: 0


In [50]:
# 过滤失败行，构建可训练数据
llm_df = pd.read_csv(OUT_LLM_CSV)
llm_ok = llm_df[llm_df["error"].isna() & llm_df["essay"].fillna("").str.len().gt(30)].copy()

human_df = df[[ID_COL, "prompt_id", PROMPT_COL, TEXT_COL, "label"]].copy()
human_df = human_df.rename(columns={TEXT_COL: "essay"})

llm_train = llm_ok[[ID_COL, "prompt_id", PROMPT_COL, "essay", "label"]].copy()
llm_train[ID_COL] = llm_train[ID_COL].astype(str) + "_llm"

mixed = pd.concat([human_df, llm_train], ignore_index=True)
mixed = mixed.drop_duplicates(subset=[PROMPT_COL, "essay", "label"]).reset_index(drop=True)
mixed.to_csv(OUT_MIXED_CSV, index=False)

print("可用 LLM 样本:", len(llm_train))
print("Human 样本:", len(human_df))
print("混合训练集:", len(mixed))
print("输出:", OUT_MIXED_CSV)
mixed["label"].value_counts()

可用 LLM 样本: 13228
Human 样本: 13266
混合训练集: 26488
输出: /Users/songling/Desktop/Artificial-Intelligence-Competition---Large-Language-Model-LLM--main/LLM-Detect AI Generated Text/CESS/CSEE_human_llm_balanced.csv


label
0    13260
1    13228
Name: count, dtype: int64

## 可选：快速质检建议

- 随机抽样 100 条，人工检查是否出现明显模板化输出。
- 统计 `essay` 长度分布，避免模型只学到“长度捷径”。
- 训练时把 CSEE 作为独立来源域，做按来源分组验证。
